# Attacker Campaigns: Clustering IPs by Behaviour 

As our data does not make it easy to infer ground-truth labels, we're dealing with an **unsupervised** learning problem. (find structure nobody labelled). The question we're trying to answer:

> Which tools / campaigns attack the honeypot, how big is each, and which networks run them?

**Unit: the attacking IP, not the session.** Command sessions are near-duplicates (in the 2026-09-16 snapshot, 55k command sessions contain only 247 distinct command sequences, and the top 20 cover 95%), so clustering sessions would just rediscover exact sequence hashes. A *campaign* is a set of IPs running the same tooling: the same credential dictionary, playbook and SSH/telnet client. Every IP with at least one login attempt is clustered; IPs that only connect (scanners) have almost no behaviour to go on and are only counted.

**Generalization, without labels.** Nothing tells us the right answer, so the test of a clustering is whether it *reproduces* on IPs, and networks, it never saw:

| split | share | used for |
|---|---|---|
| train | 60% | fitting features, embedding and clusters; stability |
| validation | 20% | choosing the algorithm and its hyperparameters |
| test | 20% | reported **once**, for the chosen configuration only |

The split is grouped by ASN (geo/'s lesson: a random-IP split leaks).

**Every run is versioned.** `pipeline.run()` hashes the config, the input files and the code into a run id and writes everything (config, metrics, tables, charts, a log) to `ml/outputs/runs/<run_id>/`. Running it again with nothing changed finds that directory and skips the work, so this notebook is cheap to rerun and always shows a run that can be reproduced exactly.

In [ ]:
import polars as pl
from IPython.display import Image

from honeypot.analysis.ml.campaigns.clustering import CampaignConfig
from honeypot.analysis.ml.campaigns.pipeline import run
from honeypot.analysis.ml.runs import RUNS_DIR, load_run, rebuild_index

## Run (or reuse) the pipeline

`CampaignConfig()` holds every hyperparameter and the grid that's tuned. Change a value (for example `CampaignConfig(kmeans_k=(6, 8, 10))`) and you get a new run id and a new run directory; the old run stays where it is. Pass `force=True` to recompute a run that already exists: its tables and charts come out byte-identical.

In [ ]:
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(120)

MODE = "offline"
config = CampaignConfig()

result = run(MODE, config=config)
loaded = load_run(result.path)
metrics, manifest = loaded["metrics"], loaded["manifest"]
charts = result.path / "charts"


def table(name: str) -> pl.DataFrame:
    return pl.read_parquet(result.path / "tables" / f"{name}.parquet")


print(f"run {result.run_id} ({'reused' if result.skipped else 'new'}): {result.path}")
print(
    f"code {manifest['code_sha256'][:12]}, git {manifest['git_sha']}, "
    f"dirty: {manifest['git_dirty']}"
)
pl.DataFrame(
    [
        {
            "input": name,
            "file": item["path"].rsplit("/", 1)[-1],
            "sha256": item["sha256"][:12],
        }
        for name, item in manifest["inputs"].items()
    ]
)

## What gets clustered

Of all source IPs, only those that tried to log in carry a behavioural fingerprint. The rest only open a connection and are left out of the clustering.

In [ ]:
pl.DataFrame([metrics["data"]])

## Features

Per IP, four blocks (`ml/campaigns/profiles.py`, tokens from `ml/tokens.py`):

- **credentials**: the `user:password` pairs and bare usernames tried (TF-IDF: a pair every botnet tries weighs little, a rare one a lot);
- **commands**: normalized commands run (C2 addresses and busybox verification tags stripped, as in the fingerprinting module);
- **client**: SSH client version, hassh, telnet option-negotiation hash;
- **behaviour**: log session and attempt counts, login success rate, active span, sessions per active hour, command / telnet / download shares.

Each block gets the same total weight, so the credential block (thousands of columns) can't drown out the dozen behaviour numbers. All of this is **fit on the training split only** (TF-IDF vocabularies, scaling, the SVD below) and only applied to validation and test, so nothing about the held-out IPs leaks into the model.

The sparse matrix is then reduced with a truncated SVD to 10-50 dimensions (a tuned hyperparameter) and each row normalized to unit length, so Euclidean distance ranks pairs like cosine similarity.

## The split

Hashing each ASN decides its split, so the same seed always gives the same split and an ASN keeps its split when later snapshots add data. The fractions hold for ASNs, not IPs, so the realized IP shares deviate a few points from 60/20/20:

In [ ]:
table("split_summary")

## Tuning on the validation split

Three algorithms that seem appropriate to investigate, each over a grid, at three embedding sizes:

- **k-means** (k): spherical clusters around centroids;
- **Ward** hierarchical clustering (k): merges the pair of clusters that least increases within-cluster variance;
- **HDBSCAN** (minimum cluster size): density-based, finds its own number of clusters and leaves outliers as noise.

Per configuration:

- on **train**: silhouette (how much closer each IP is to its own cluster than to the next), and **stability**, the mean adjusted Rand index between the full-train clustering and reclusterings of random 80% subsamples (1 = identical every time);
- on **validation**: **prediction strength** (Tibshirani & Walther 2005). The validation IPs are clustered on their own, and separately assigned to the nearest *train* centroid. For each validation cluster: the share of its pairs of IPs that the train model also keeps together. If the clusters are real structure, both views agree.

Tibshirani & Walther use the minimum over clusters. That's reported (`val_prediction_strength`), but selection uses the size-weighted mean: on this data the minimum is always set by one heterogeneous leftover cluster, so it can't tell configurations apart. Which individual campaigns don't reproduce is shown per campaign on the test split instead.

**Selection rule**: among configurations with train stability >= 0.8 and weighted validation prediction strength >= 0.8, take the highest prediction strength; configurations within 0.02 of it count as tied, and the tie goes to the best validation silhouette, then fewer clusters. (Preferring *more* clusters on ties over-split a planted campaign in the test fixture: prediction strength can't tell a campaign from a consistent sub-split of it.)

In [ ]:
Image(charts / "tuning_curves.png")

In [ ]:
tuning = table("tuning")
tuning.sort("val_prediction_strength_weighted", descending=True).select(
    "chosen",
    "algorithm",
    "param",
    "n_components",
    "train_n_clusters",
    "train_stability",
    "train_silhouette",
    "val_silhouette",
    "val_prediction_strength",
    "val_prediction_strength_weighted",
).head(12)

In [ ]:
chosen = metrics["chosen"]
print(
    f"chosen: {chosen['algorithm']} with param {chosen['param']} on "
    f"{chosen['n_components']} SVD dimensions "
    f"({chosen['explained_variance']:.0%} of variance) -- {chosen['rule']}"
)

## The test split, once

The chosen configuration is refit on train + validation, and only now is the test split touched: each test IP is assigned to its nearest campaign centroid, and the test IPs are also clustered on their own with the same algorithm. If validation and test agree, the selection didn't overfit the validation split.

In [ ]:
pl.DataFrame(
    [
        {
            "split": "validation",
            "prediction_strength_weighted": chosen["val_prediction_strength_weighted"],
            "prediction_strength_min": chosen["val_prediction_strength"],
            "silhouette": chosen["val_silhouette"],
        },
        {
            "split": "test",
            "prediction_strength_weighted": metrics["test"][
                "prediction_strength_weighted"
            ],
            "prediction_strength_min": metrics["test"]["prediction_strength"],
            "silhouette": metrics["test"]["silhouette_assigned"],
        },
    ]
)

Per campaign, **test reproducibility** is the share of pairs of test IPs assigned to the campaign that the independent test clustering also keeps together. Below 0.8 a campaign is flagged: it's more likely a catch-all of leftover IPs than one tool.

In [ ]:
Image(charts / "campaign_sizes.png")

## Does it match structure found independently?

Two labellings exist that the clustering never saw directly:

- **ch. 4's credential-graph clusters**: IPs linked by sharing rare credential pairs (network/graph.py). Most IPs are singletons there, so ARI between the two partitions is near zero by construction; the meaningful number is how many of the IPs ch. 4 *does* link end up in one campaign.
- **each IP's dominant command sequence**: IPs replaying the exact same playbook.

Neither is ground truth (credentials and commands are also feature blocks), so this is a sanity check, measured on the test split only.

In [ ]:
Image(charts / "external_agreement.png")

In [ ]:
table("external_agreement")

## The campaign catalog

One row per campaign: size, the splits its IPs came from, the most common usernames, credential pairs, commands and client fingerprints, the networks and countries, and behavioural shares. Command text has every IPv4 address replaced by `<IP>`.

In [ ]:
catalog = table("catalog").sort("campaign")
catalog.select(
    "campaign",
    "ips",
    "sessions",
    "asns",
    "test_reproducibility",
    "top_usernames",
    "top_commands",
    "top_clients",
    "top_asns",
    "top_countries",
)

In [ ]:
Image(charts / "campaign_profiles.png")

In [ ]:
Image(charts / "campaign_map.png")

The map is a t-SNE projection of the embedding: IPs close together behave alike, but distances between far-apart groups (and group sizes) mean nothing. Gray points belong to the smaller campaigns.

Timing separates steady background campaigns from bursts:

In [ ]:
Image(charts / "campaign_activity.png")

## Run history

Every finished run, with its headline metrics, from `ml/outputs/runs/index.parquet` (rebuilt from the run directories after every run).

In [ ]:
index = rebuild_index(RUNS_DIR)
index.select(
    "run_id",
    "finished_at",
    "input.snapshot",
    "metrics.chosen.algorithm",
    "metrics.chosen.param",
    "metrics.test.prediction_strength_weighted",
    "metrics.campaigns.count",
    "metrics.campaigns.reproducible",
)

## Caveats

- **An IP isn't an actor.** Residential IPs churn and NAT/CGNAT puts many devices behind one address; a campaign is "IPs that behaved alike", not one operator. Conversely one operator can run several tools and show up in several campaigns.
- **Catch-all clusters.** k-means assigns every IP somewhere, so campaigns flagged as not reproducing collect leftovers; read them as "other", not as a tool.
- **One deployment, one month.** Two sensors, 2026-08-17 to 2026-09-16. The grouped split shows the campaigns reproduce on unseen networks within that window, not that they'd look the same next month.
- **Unsupervised means unverified.** Agreement with ch. 4 and the command sequences supports the campaigns, but nobody has labelled them; the names in the README are interpretations of the catalog.